# Parte 3 - Cruce por ubigeo y análisis

### Responsable: Fiorella Ampuero

## 0. Librerías y rutas

Ejecutar este notebook desde la carpeta `assignment_2/`. Se asume que ya existen `datos/decretos_por_departamento.csv` y `datos/lluvias_por_departamento.csv` (25 filas cada uno).


In [2]:
import unicodedata
from pathlib import Path

import pandas as pd
import plotly.express as px

Path("datos").mkdir(exist_ok=True)
print("pandas:", pd.__version__)
try:
    import plotly
    print("plotly:", plotly.__version__)
except Exception as e:
    print(e)


pandas: 2.3.3
plotly: 7.1.0


## 1. Crear `datos/ubigeos.csv`

Usamos la tabla oficial del INEI del enunciado en el issue. El ubigeo se guarda como **texto de 2 dígitos** (`01`, no `1`) y el archivo en `utf-8-sig` para que las tildes (Áncash, Apurímac, Huánuco, Junín, San Martín) se lean bien en Windows.


In [4]:
ubigeos = pd.DataFrame({
    "ubigeo": [f"{i:02d}" for i in range(1, 26)],
    "departamento": ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                     "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                     "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                     "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                     "San Martín", "Tacna", "Tumbes", "Ucayali"],
})
# Asegurar texto de 2 dígitos aunque el código se edite con enteros
ubigeos["ubigeo"] = ubigeos["ubigeo"].astype(str).str.zfill(2)
ubigeos.to_csv("datos/ubigeos.csv", index=False, encoding="utf-8-sig")
assert len(ubigeos) == 25
assert ubigeos["ubigeo"].str.fullmatch(r"\d{2}").all()
assert ubigeos["departamento"].is_unique
ubigeos


,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho
5,06,Cajamarca
6,07,Callao
7,08,Cusco
8,09,Huancavelica
9,10,Huánuco


## 2. Leer `ubigeos.csv` de dos formas: con y sin `dtype`

Comparamos `pd.read_csv` normal contra `pd.read_csv(..., dtype={"ubigeo": str})`.


In [5]:
normal = pd.read_csv("datos/ubigeos.csv")  # sin dtype: infiere el tipo
texto = pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})  # forzamos texto

print("--- Sin dtype ---")
print(normal.dtypes.to_string())
print(normal.head(8).to_string(index=False))
print("\n--- Con dtype str ---")
print(texto.dtypes.to_string())
print(texto.head(8).to_string(index=False))
print("\nPrimer ubigeo:", repr(normal.loc[0, "ubigeo"]), "vs", repr(texto.loc[0, "ubigeo"]))


--- Sin dtype ---
ubigeo           int64
departamento    object
 ubigeo departamento
      1     Amazonas
      2       Áncash
      3     Apurímac
      4     Arequipa
      5     Ayacucho
      6    Cajamarca
      7       Callao
      8        Cusco

--- Con dtype str ---
ubigeo          object
departamento    object
ubigeo departamento
    01     Amazonas
    02       Áncash
    03     Apurímac
    04     Arequipa
    05     Ayacucho
    06    Cajamarca
    07       Callao
    08        Cusco

Primer ubigeo: np.int64(1) vs '01'


### Por qué importa leer el ubigeo como texto

Sin `dtype`, pandas infiere que `01, 02, ...` son **números** y los convierte a entero (`01` se vuelve `1`): se pierde el cero inicial. Eso rompe el cruce porque (1) la llave ya no coincide con el código oficial de 2 dígitos, (2) un `merge` entre una columna `int` (1) y una `str` ("01") no empareja nada o produce nulos, y (3) el archivo final queda mal (`1` en vez de `01`). Con `dtype={"ubigeo": str}` el código se conserva como texto `"01"`, el `merge` por ubigeo funciona y `datos/tabla_final.csv` cumple lo pedido. Por eso, desde aquí usamos siempre `dtype={"ubigeo": str}` y `str.zfill(2)` al guardar.


## 3. Agregar el ubigeo a cada tabla

Cruzamos por **nombre de departamento** solo para traer el ubigeo; el cruce analítico final (paso 5) será **por ubigeo**. Leemos el ubigeo como texto.


In [6]:
ubi = pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})
ubi["ubigeo"] = ubi["ubigeo"].str.zfill(2)

decretos = pd.read_csv("datos/decretos_por_departamento.csv", encoding="utf-8-sig")
lluvias = pd.read_csv("datos/lluvias_por_departamento.csv", encoding="utf-8-sig")
print("decretos:", decretos.shape, "| lluvias:", lluvias.shape, "| ubigeos:", ubi.shape)

dec_ub = decretos.merge(ubi, on="departamento", how="left")
llu_ub = lluvias.merge(ubi, on="departamento", how="left")
print("\nDecretos + ubigeo:")
print(dec_ub.head(10).to_string(index=False))
print("\nLluvias + ubigeo:")
print(llu_ub.head(10).to_string(index=False))


decretos: (25, 3) | lluvias: (25, 6) | ubigeos: (25, 2)

Decretos + ubigeo:
departamento  declaratorias  prorrogas ubigeo
    Amazonas              1          0     01
      Áncash              0          0     02
    Apurímac              0          0     03
    Arequipa              0          0     04
    Ayacucho              1          0     05
   Cajamarca              0          0     06
      Callao              0          0     07
       Cusco              0          0     08
Huancavelica              0          0     09
     Huánuco              0          0     10

Lluvias + ubigeo:
departamento      capital   latitud  longitud  lluvia_total_mm  dias_lluvia_fuerte ubigeo
    Amazonas  Chachapoyas  -6.23169 -77.86903            436.5                   1     01
      Áncash       Huaraz  -9.52614 -77.52869            975.3                   7     02
    Apurímac      Abancay -13.63426 -72.88380            711.8                   2     03
    Arequipa     Arequipa -16.39899 -71

## 4. Verificación: departamentos sin ubigeo

Si un nombre viene con tilde distinta, sin tilde u otro texto (`Ancash` vs `Áncash`), el `merge` exacto deja `ubigeo = NaN`. Los listamos en ambas tablas y, de haber alguno, lo corregimos con un mapa normalizado (minúsculas + sin tildes) hasta que no quede ninguno.


In [7]:
def normalizar(s):
    s = str(s).strip().lower()
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

print("Sin ubigeo en decretos:", dec_ub[dec_ub["ubigeo"].isna()]["departamento"].tolist() or "ninguno")
print("Sin ubigeo en lluvias:", llu_ub[llu_ub["ubigeo"].isna()]["departamento"].tolist() or "ninguno")

# Mapa normalizado como red de seguridad: detecta diferencias de tildes/mayúsculas
mapa = {normalizar(d): u for d, u in zip(ubi["departamento"], ubi["ubigeo"])}
for nombre, df in [("decretos", dec_ub), ("lluvias", llu_ub)]:
    for d in df[df["ubigeo"].isna()]["departamento"].tolist():
        print(nombre, "sin match exacto:", repr(d), "-> normalizado:", repr(normalizar(d)), "ubigeo:", mapa.get(normalizar(d)))

# Corrección: si quedara algún NaN, se rellena con el mapa normalizado
dec_ub["ubigeo"] = dec_ub["ubigeo"].fillna(dec_ub["departamento"].map(lambda d: mapa.get(normalizar(d))))
llu_ub["ubigeo"] = llu_ub["ubigeo"].fillna(llu_ub["departamento"].map(lambda d: mapa.get(normalizar(d))))
assert dec_ub["ubigeo"].notna().all(), "Quedan departamentos sin ubigeo en decretos"
assert llu_ub["ubigeo"].notna().all(), "Quedan departamentos sin ubigeo en lluvias"
print("\nVerificación OK: los 25 departamentos de cada tabla tienen su ubigeo (0 pendientes).")


Sin ubigeo en decretos: ninguno
Sin ubigeo en lluvias: ninguno

Verificación OK: los 25 departamentos de cada tabla tienen su ubigeo (0 pendientes).


## 5. Cruce de las dos tablas por ubigeo (no por nombre)

Partimos de `ubigeos` (25 filas) y hacemos `left merge` con cada tabla. Así ningún departamento desaparece aunque tenga 0 declaratorias o falte en alguna fuente.


In [8]:
final = (ubi
         .merge(dec_ub[["ubigeo", "declaratorias", "prorrogas"]], on="ubigeo", how="left")
         .merge(llu_ub[["ubigeo", "capital", "latitud", "longitud", "lluvia_total_mm", "dias_lluvia_fuerte"]], on="ubigeo", how="left"))
# Orden oficial por ubigeo
final = final.sort_values("ubigeo").reset_index(drop=True)
print(final.to_string(index=False))


ubigeo  departamento  declaratorias  prorrogas          capital   latitud  longitud  lluvia_total_mm  dias_lluvia_fuerte
    01      Amazonas              1          0      Chachapoyas  -6.23169 -77.86903            436.5                   1
    02        Áncash              0          0           Huaraz  -9.52614 -77.52869            975.3                   7
    03      Apurímac              0          0          Abancay -13.63426 -72.88380            711.8                   2
    04      Arequipa              0          0         Arequipa -16.39899 -71.53747            189.9                   0
    05      Ayacucho              1          0         Ayacucho -13.16380 -74.22345            485.1                   0
    06     Cajamarca              0          0        Cajamarca  -7.16378 -78.50027            773.3                   1
    07        Callao              0          0           Callao -12.05162 -77.13452             39.2                   0
    08         Cusco            

## 6. Verificación: 25 filas y ceros donde no hubo declaratorias

Los departamentos sin ninguna declaratoria deben aparecer con `0`, no desaparecer. Por eso se parte de `ubigeos` con `how="left"` y se rellenan los nulos con 0 (un `inner` los habría eliminado si alguna fuente viniera incompleta).


In [9]:
final["declaratorias"] = final["declaratorias"].fillna(0).astype(int)
final["prorrogas"] = final["prorrogas"].fillna(0).astype(int)
assert len(final) == 25, f"Debe tener 25 filas, tiene {len(final)}"
assert final["ubigeo"].is_unique, "Ubigeo repetido"
assert final["ubigeo"].str.fullmatch(r"\d{2}").all(), "Ubigeo no es texto de 2 dígitos"
print("Filas:", len(final))
print("Declaratorias totales:", int(final["declaratorias"].sum()), "| Prórrogas totales:", int(final["prorrogas"].sum()))
print("Departamentos con 0 declaratorias:", int((final["declaratorias"] == 0).sum()))
print(final[["ubigeo", "departamento", "declaratorias", "lluvia_total_mm"]].to_string(index=False))


Filas: 25
Declaratorias totales: 3 | Prórrogas totales: 0
Departamentos con 0 declaratorias: 22
ubigeo  departamento  declaratorias  lluvia_total_mm
    01      Amazonas              1            436.5
    02        Áncash              0            975.3
    03      Apurímac              0            711.8
    04      Arequipa              0            189.9
    05      Ayacucho              1            485.1
    06     Cajamarca              0            773.3
    07        Callao              0             39.2
    08         Cusco              0            710.7
    09  Huancavelica              0            803.8
    10       Huánuco              0            444.8
    11           Ica              0             25.1
    12         Junín              0            910.4
    13   La Libertad              0             98.9
    14    Lambayeque              0             98.8
    15          Lima              0             13.9
    16        Loreto              0           1581.8
   

## 7. Guardar `datos/tabla_final.csv`

El ubigeo se guarda como texto de 2 dígitos (`01`, no `1`) y se re-lee con `dtype=str` para comprobarlo.


In [10]:
final["ubigeo"] = final["ubigeo"].astype(str).str.zfill(2)
final.to_csv("datos/tabla_final.csv", index=False, encoding="utf-8-sig")
check = pd.read_csv("datos/tabla_final.csv", dtype={"ubigeo": str})
assert len(check) == 25
assert check["ubigeo"].str.fullmatch(r"\d{2}").all(), "El ubigeo no quedó como texto de 2 dígitos"
print(check.dtypes.to_string())
print(check.head(10).to_string(index=False))
print("\nGuardado OK: datos/tabla_final.csv con", len(check), "filas")


ubigeo                 object
departamento           object
declaratorias           int64
prorrogas               int64
capital                object
latitud               float64
longitud              float64
lluvia_total_mm       float64
dias_lluvia_fuerte      int64
ubigeo departamento  declaratorias  prorrogas      capital   latitud  longitud  lluvia_total_mm  dias_lluvia_fuerte
    01     Amazonas              1          0  Chachapoyas  -6.23169 -77.86903            436.5                   1
    02       Áncash              0          0       Huaraz  -9.52614 -77.52869            975.3                   7
    03     Apurímac              0          0      Abancay -13.63426 -72.88380            711.8                   2
    04     Arequipa              0          0     Arequipa -16.39899 -71.53747            189.9                   0
    05     Ayacucho              1          0     Ayacucho -13.16380 -74.22345            485.1                   0
    06    Cajamarca              0

## 8. Gráficos con Plotly

Dos gráficos: (a) dispersión lluvia vs declaratorias con el nombre del departamento en cada punto; (b) barras con los 10 departamentos con más declaratorias.


In [11]:
fig1 = px.scatter(final, x="lluvia_total_mm", y="declaratorias", text="departamento",
                 hover_data=["ubigeo", "departamento", "lluvia_total_mm", "declaratorias", "dias_lluvia_fuerte"],
                 title="Lluvia acumulada vs declaratorias de emergencia por lluvias (ene-may 2022)",
                 labels={"lluvia_total_mm": "Lluvia total en la capital (mm)", "declaratorias": "Declaratorias"})
fig1.update_traces(textposition="top center")
fig1.update_layout(yaxis=dict(dtick=1))
fig1.show()


In [12]:
top10 = final.sort_values(["declaratorias", "lluvia_total_mm"], ascending=[False, False]).head(10)
print(top10[["ubigeo", "departamento", "declaratorias", "lluvia_total_mm"]].to_string(index=False))
fig2 = px.bar(top10.sort_values("declaratorias", ascending=True), x="declaratorias", y="departamento", orientation="h",
              hover_data=["ubigeo", "lluvia_total_mm", "dias_lluvia_fuerte"],
              title="Top 10 departamentos con más declaratorias por lluvias (ene-may 2022)",
              labels={"declaratorias": "Declaratorias", "departamento": "Departamento"})
fig2.show()


ubigeo  departamento  declaratorias  lluvia_total_mm
    05      Ayacucho              1            485.1
    01      Amazonas              1            436.5
    20         Piura              1             89.2
    16        Loreto              0           1581.8
    25       Ucayali              0           1288.8
    02        Áncash              0            975.3
    12         Junín              0            910.4
    17 Madre de Dios              0            908.0
    09  Huancavelica              0            803.8
    22    San Martín              0            789.8


### Lectura rápida de los datos (lo que muestran las tablas y gráficos)

- Declaratorias por lluvias en la temporada: **3 en total** (Amazonas 1, Ayacucho 1, Piura 1; el resto 0; prórrogas 0). Hay un **empate triple** en el primer lugar.
- Lluvia acumulada (capital, ene-may 2022): Loreto 1581.8 mm, Ucayali 1288.8 mm, Áncash 975.3 mm, Junín 910.4 mm, Madre de Dios 908.0 mm. Piura (89.2 mm), Ayacucho (485.1 mm) y Amazonas (436.5 mm) están muy por debajo.
- El diagrama de dispersión muestra los 3 puntos con `y = 1` a la **izquierda** (poca lluvia) y los puntos con más lluvia en `y = 0`: no hay relación positiva visible.


### 9a. ¿Cuáles fueron los 3 departamentos con más declaratorias? ¿Coinciden con los 3 donde más llovió?

Los 3 con más declaratorias fueron **Amazonas, Ayacucho y Piura, con 1 cada uno** (empate triple; ningún otro departamento tuvo declaratorias por lluvias en la muestra y no hubo prórrogas). Los 3 donde más llovió (medido en la capital) fueron **Loreto (1581.8 mm), Ucayali (1288.8 mm) y Áncash (975.3 mm)**. **No coinciden en ninguno**: los tres más lluviosos tuvieron 0 declaratorias y los tres con declaratoria están lejos del top de lluvia (Piura apenas 89.2 mm).


### 9b. ¿Hay departamentos con mucha lluvia y pocas o ninguna declaratoria, o al revés? Den al menos dos razones posibles.

Sí, en ambos sentidos. Con mucha lluvia y 0 declaratorias: **Loreto (1581.8 mm, 25 días de 20 mm o más), Ucayali (1288.8 mm, 20 días fuertes), Áncash, Junín y Madre de Dios (unos 908-975 mm)**. Al revés, con poca lluvia y 1 declaratoria: **Piura (89.2 mm, 0 días fuertes)** y en menor medida Amazonas (436.5 mm) y Ayacucho (485.1 mm). Posibles razones: (1) **la lluvia en la capital no representa al departamento**: un decreto puede responder a lluvias localizadas en otras provincias (por ejemplo, sierra de Piura) mientras la capital costera casi no llueve, y viceversa; (2) **la declaratoria no depende solo de cuántos mm caen**, sino del daño y la vulnerabilidad (deslizamientos, desbordes, viviendas expuestas, suelo ya saturado) y de criterios administrativos (un solo DS N.° 032-2022-PCM agrupó a tres departamentos a la vez); además un total acumulado no distingue intensidad, duración ni recurrencia.


### 9c. ¿Qué porcentaje fue por peligro inminente y qué porcentaje por impacto de daños? ¿Qué dice eso sobre la gestión del riesgo?

En `datos/decretos_lluvias.csv` hay **1 decreto de lluvia (DS N.° 032-2022-PCM, tipo declara)**: **0 % por peligro inminente (0 de 1) y 100 % por impacto de daños (1 de 1)**. Es decir, en esta temporada y con este procedimiento de búsqueda, toda la declaratoria por lluvias llegó **después del daño** y ninguna antes como prevención. Sugiere una gestión predominantemente **reactiva** en la muestra: el Estado actuó para atender distritos ya afectados y no (al menos vía DS-PCM captados aquí) para anticiparse a un peligro pronosticado. El dato debe leerse con cautela porque n = 1: basta un decreto para mover el porcentaje de 0 a 100, así que no generaliza a toda la gestión del riesgo del país.


### 9d. Tres limitaciones de este análisis

1. **La lluvia es de un solo punto (la capital)**: no representa la variabilidad dentro del departamento (costa/sierra/selva, altitud, microclimas) ni la celda del modelo; un promedio o varias estaciones por departamento sería más fiel. 2. **Muestra de decretos mínima y atada a un buscador**: n = 1 decreto de lluvia (3 declaratorias idénticas por un DS multidepartamental); el conteo depende de los filtros del buscador de gob.pe (término, PCM, fechas) y puede omitir normas con otra redacción o publicadas fuera de rango. 3. **Conteo grueso por departamento**: cada departamento suma 1 por aparecer en el DS, sin distinguir cuántos distritos/provincias, población afectada, severidad del daño ni duración; correlación lluvia-declaratorias no implica causalidad y con tantos ceros (22 de 25) ningún gráfico prueba una relación estadística.


### Conclusión

En la temporada enero-mayo de 2022, **el Estado no declaró emergencia donde más llovió**. Las únicas declaratorias por lluvias fueron un decreto que agrupó a Amazonas, Ayacucho y Piura (1 cada uno), mientras los departamentos más lluviosos (Loreto, Ucayali, Áncash) no tuvieron ninguna y Piura, casi sin lluvia en su capital, sí. La única declaratoria fue 100 % por impacto de daños, o sea reactiva. Con una muestra tan pequeña y midiendo la lluvia solo en la capital, no se puede afirmar un patrón general, pero en este periodo la emergencia respondió al daño localizado y a la vulnerabilidad, no al ranking de milímetros.
